# **Naive RAG**

Naive RAG is a basic Retrieval-Augmented Generation approach that retrieves relevant information from a knowledge source and provides it to an LLM to generate a context-aware response.

**Install libraries**

In [1]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-groq \
    pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 

**Get the APIKEY**

In [3]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


**PDF upload**

In [4]:
from google.colab import files

uploaded = files.upload()

Saving rag_sample_data_structure.pdf to rag_sample_data_structure.pdf


**Load PDF**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

/tmp/ipykernel_1004/345259269.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF loaded successfully!
Number of pages: 80


**Chunking**

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunking completed!")
print("Number of chunks:", len(chunks))

Chunking completed!
Number of chunks: 267


**Embedding**

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

chunk_embeddings = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Embeddings created!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created!


**Vectore database**

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector database created!")

Vector database created!


**Create Retriever**

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

**Ask Question**

In [10]:
question = input("Ask: ")

Ask: what is algorithm


**Retrieve Relevant Information**

In [11]:
docs = retriever.invoke(question)

for doc in docs:
    print(doc.page_content)

Lecture 1 - Introduction to Design and analysis of  algorithms 
 
What is an algorithm?        
 Algorithm is a set of steps to complete a task.     
 For example,            
 Task: to make a cup of tea.        
 Algorithm: 
• add water and milk to the kettle,  
• boilit, add tea leaves, 
• Add sugar, and then serve it in cup. 
What is Computer algorithm ? 
‘’a set of steps to accomplish or complete a task t hat is described precisely enough that a 
computer can run it’’.
algorithm; this also results in a minimum cost tree; this algorithm is called Krusleat’s algorithm. 
Prim’s Algorithm Prim’s Algorithm Prim’s Algorithm Prim’s Algorithm
computer can run it’’. 
Described precisely : very difficult for a machine to know how much wat er, milk to be added 
etc. in the above tea making algorithm. 
These algorithmsrun on computers or computational d evices.Forexample, GPS in our 
smartphones, Google hangouts. 
GPS uses shortest path algorithm. Online shopping uses cryptography which uses R

**Connect Groq**

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    api_key=GROQ_API_KEY
)

**Give Context to LLM**

In [16]:
context = "\n\n".join(
    doc.page_content for doc in docs
)

prompt = f"""
Answer using only this context:

{context}

Question:
{question}
"""

**Generate Answer**

In [17]:
answer = llm.invoke(prompt)

print(answer.content)

An algorithm is a set of steps to complete a task.
